# 10 — Lemmatization in Natural Language Processing

---

## 1. Overview

Lemmatization is an NLP preprocessing technique that converts a word into its meaningful dictionary base form, called a *lemma*. Unlike stemming, lemmatization considers linguistic information such as vocabulary and part of speech.

For example, `playing`, `played`, and `plays` can be reduced to `play`. Similarly, the irregular plural `children` is reduced to `child`.

> **Notebook structure:** Part I uses spaCy. Part II uses NLTK. The final practical section reads the companion file `alice_extract.txt` and processes the story extract.

## 2. Stemming vs. lemmatization

Both techniques normalize related word forms, but their methods and outputs differ.

| Aspect | Stemming | Lemmatization |
|---|---|---|
| Method | Removes word endings using rules | Uses vocabulary and linguistic analysis |
| Result | May not be a dictionary word | Usually a valid base word |
| Part of speech | Generally not considered | Can be considered |
| Example | `studies` → `studi` | `studies` → `study` |

**Why lemmatization matters:** It helps reduce different grammatical forms of a word to a common meaningful form, which can improve consistency in tasks such as text classification, search, and information retrieval.

## 3. Environment setup

Install the libraries used in this notebook. If they are already installed, you can skip the installation commands.

- **spaCy:** tokenization, part-of-speech tagging, and lemmatization.
- **NLTK:** WordNet-based lemmatization.

In [1]:
# Install the NLP libraries.
%pip install spacy nltk

In [2]:
# Download spaCy's small English language model.
# Run this command once in the active Python environment.
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 6.1 MB/s  0:00:02
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Part I — Lemmatization using spaCy

## 4. Import spaCy and load the language model

The English model provides linguistic annotations that spaCy uses to determine each token's lemma. Load the model once and reuse the `nlp` object throughout this section.

In [3]:
# Import spaCy and load its small English language model.
import spacy

nlp = spacy.load("en_core_web_sm")

## 5. Basic lemmatization

### 5.1 Lemmatizing individual words

Pass a sentence to the language pipeline. spaCy returns a `Doc` object containing tokens, and `token.lemma_` provides the lemma for each token.

In [4]:
# Example 1: Convert each token to its lemma.
text = "The boys are playing in the gardens."

doc = nlp(text)

for token in doc:
    print(f"{token.text:<12} -> {token.lemma_}")

The          -> the
boys         -> boy
are          -> be
playing      -> play
in           -> in
the          -> the
gardens      -> garden
.            -> .


### 5.2 Lemmatizing plural nouns

Plural nouns are often converted into their singular base forms. Observe how spaCy processes each noun in the example.

In [5]:
# Example 2: Lemmatize plural nouns.
text = "Cats dogs horses cars"

doc = nlp(text)

for token in doc:
    print(f"{token.text:<12} -> {token.lemma_}")

Cats         -> cat
dogs         -> dog
horses       -> horse
cars         -> car


### 5.3 Lemmatizing verb forms

Different grammatical forms of a verb can share the same lemma.

In [6]:
# Example 3: Compare different forms of the verb "play".
text = "playing played plays"

doc = nlp(text)

for token in doc:
    print(f"{token.text:<12} -> {token.lemma_}")

playing      -> play
played       -> play
plays        -> play


### 5.4 Lemmatizing irregular verb forms

Lemmatization can also handle irregular verb forms when the language model has enough linguistic information.

In [7]:
# Example 4: Lemmatize verb forms in context.
text = "I am eating. She ate dinner. They have eaten food."

doc = nlp(text)

for token in doc:
    print(f"{token.text:<12} -> {token.lemma_}")

I            -> I
am           -> be
eating       -> eat
.            -> .
She          -> she
ate          -> eat
dinner       -> dinner
.            -> .
They         -> they
have         -> have
eaten        -> eat
food         -> food
.            -> .


### 5.5 Lemmatization of a complete sentence

The lemmas can be collected into a list and joined to form a normalized version of the sentence. This example keeps all tokens, including stop words and punctuation, to make the transformation visible.

In [8]:
# Example 5: Reconstruct a sentence using token lemmas.
text = "The students were studying their lessons."

doc = nlp(text)

# Extract the lemma of every token and join the results.
lemmatized_text = " ".join(token.lemma_ for token in doc)

print("Original sentence:")
print(text)

print("\nLemmatized sentence:")
print(lemmatized_text)

Original sentence:
The students were studying their lessons.

Lemmatized sentence:
the student be study their lesson .


### 5.6 Lemmatizing irregular words

Words such as `mice`, `children`, and `went` do not follow simple suffix-removal patterns. This example demonstrates spaCy's handling of such forms.

In [9]:
# Example 6: Examine irregular and comparative forms.
text = "good better best went gone mice children"

doc = nlp(text)

for token in doc:
    print(f"{token.text:<12} -> {token.lemma_}")

good         -> good
better       -> well
best         -> well
went         -> went
gone         -> go
mice         -> mice
children     -> child


### 5.7 Lemmatization with part-of-speech information

spaCy assigns each token a part-of-speech label. The `pos_` attribute displays the label, while `lemma_` displays the corresponding lemma.

In [10]:
# Example 7: Display each token, its part of speech, and its lemma.
text = "The children are running quickly."

doc = nlp(text)

for token in doc:
    print(
        f"Token: {token.text:<12} "
        f"POS: {token.pos_:<8} "
        f"Lemma: {token.lemma_}"
    )

Token: The          POS: DET      Lemma: the
Token: children     POS: NOUN     Lemma: child
Token: are          POS: AUX      Lemma: be
Token: running      POS: VERB     Lemma: run
Token: quickly      POS: ADV      Lemma: quickly
Token: .            POS: PUNCT    Lemma: .


## 6. Text cleaning with lemmatization

### 6.1 Remove punctuation

Punctuation can be excluded with the `is_punct` attribute. The remaining tokens are converted to their lemmas.

In [11]:
# Example 8: Remove punctuation and lemmatize the remaining tokens.
text = "Dogs are barking, cats are sleeping!"

doc = nlp(text)

lemmas = []

for token in doc:
    if not token.is_punct:
        lemmas.append(token.lemma_)

print(lemmas)

['dog', 'be', 'bark', 'cat', 'be', 'sleep']


### 6.2 Remove stop words and punctuation

Stop words are common words such as `the`, `are`, and `is`. They may be removed when the goal is to focus on the more informative words in a text. spaCy identifies stop words using `is_stop`.

In [12]:
# Example 9: Exclude stop words and punctuation before lemmatizing.
text = "The students are learning Artificial Intelligence."

doc = nlp(text)

lemmas = []

for token in doc:
    if not token.is_stop and not token.is_punct:
        lemmas.append(token.lemma_)

print(lemmas)

['student', 'learn', 'Artificial', 'Intelligence']


### 6.3 Create a reusable lemmatization function

A function makes the preprocessing logic reusable. The function below removes stop words and punctuation by default and returns the remaining lemmas as a single string.

In [13]:
# Example 10: Define a reusable function for text lemmatization.
def lemmatize_text(text):
    """Return text with stop words and punctuation removed and words lemmatized."""

    doc = nlp(text)
    result = []

    for token in doc:
        if not token.is_stop and not token.is_punct:
            result.append(token.lemma_)

    return " ".join(result)


# Test the function with a sample sentence.
sample_text = "The students were studying different subjects."

processed_text = lemmatize_text(sample_text)

print("Original text:")
print(sample_text)

print("\nProcessed text:")
print(processed_text)

Original text:
The students were studying different subjects.

Processed text:
student study different subject


# Part II — Lemmatization using NLTK

## 7. Prepare the NLTK environment

NLTK's `WordNetLemmatizer` uses the WordNet lexical database. Download the required WordNet resources before using the lemmatizer.

In [14]:
# Install NLTK if it is not already installed.
%pip install nltk

In [15]:
# Download the WordNet database and multilingual WordNet data.
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [16]:
# Import and initialize the WordNet lemmatizer.
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

## 8. Basic NLTK lemmatization

### 8.1 Lemmatizing nouns

By default, `WordNetLemmatizer` treats a word as a noun. The following example converts common plural nouns into their base forms.

In [17]:
# Example 1: Lemmatize plural nouns using the default noun setting.
words = ["cats", "dogs", "cars", "houses"]

for word in words:
    print(f"{word:<12} -> {lemmatizer.lemmatize(word)}")

cats         -> cat
dogs         -> dog
cars         -> car
houses       -> house


### 8.2 Lemmatizing verbs

For verbs, specify `pos="v"`. Without this argument, NLTK assumes the word is a noun, which may produce a different result.

In [18]:
# Example 2: Lemmatize different verb forms.
verbs = ["playing", "played", "plays"]

for verb in verbs:
    print(f"{verb:<12} -> {lemmatizer.lemmatize(verb, pos='v')}")

playing      -> play
played       -> play
plays        -> play


### 8.3 Lemmatizing irregular nouns

WordNet can map irregular plural nouns to their singular forms.

In [19]:
# Example 3: Lemmatize irregular plural nouns.
words = ["children", "mice", "geese", "men", "women"]

for word in words:
    print(f"{word:<12} -> {lemmatizer.lemmatize(word, pos='n')}")

children     -> child
mice         -> mouse
geese        -> goose
men          -> men
women        -> woman


### 8.4 Lemmatizing adjectives

Use `pos="a"` to specify that the input words should be treated as adjectives.

In [20]:
# Example 4: Lemmatize adjective forms.
words = ["better", "best", "faster", "happier"]

for word in words:
    print(f"{word:<12} -> {lemmatizer.lemmatize(word, pos='a')}")

better       -> good
best         -> best
faster       -> fast
happier      -> happy


### 8.5 Lemmatizing adverbs

Use `pos="r"` for adverbs.

In [21]:
# Example 5: Lemmatize adverbs.
words = ["quickly", "slowly", "happily"]

for word in words:
    print(f"{word:<12} -> {lemmatizer.lemmatize(word, pos='r')}")

quickly      -> quickly
slowly       -> slowly
happily      -> happily


### 8.6 Lemmatizing irregular verbs

This example demonstrates how WordNet processes irregular verb forms when the verb part of speech is specified.

In [22]:
# Example 6: Lemmatize irregular verb forms.
words = ["went", "gone", "going", "eaten", "ate"]

for word in words:
    print(f"{word:<12} -> {lemmatizer.lemmatize(word, pos='v')}")

went         -> go
gone         -> go
going        -> go
eaten        -> eat
ate          -> eat


### 8.7 Compare noun and verb lemmatization

The same spelling can produce different lemmas depending on its grammatical role. Compare the word `meeting` as a noun and as a verb.

In [23]:
# Example 7: Compare the noun and verb interpretations of one word.
word = "meeting"

noun_lemma = lemmatizer.lemmatize(word, pos="n")
verb_lemma = lemmatizer.lemmatize(word, pos="v")

print("As a noun:", noun_lemma)
print("As a verb:", verb_lemma)

As a noun: meeting
As a verb: meet


### 8.8 Lemmatizing a list of words

The following example selects a part of speech using a simple suffix-based condition. This is a demonstration only: suffix checks do not reliably identify the grammatical role of every word.

In [24]:
# Example 8: Apply a basic POS choice to a list of words.
words = [
    "studying",
    "studies",
    "studied",
    "students",
    "books"
]

for word in words:
    # Use verb lemmatization for common verb endings.
    if word.endswith(("ing", "ed")):
        lemma = lemmatizer.lemmatize(word, pos="v")
    else:
        lemma = lemmatizer.lemmatize(word, pos="n")

    print(f"{word:<12} -> {lemma}")

studying     -> study
studies      -> study
studied      -> study
students     -> student
books        -> book


### 8.9 Complete NLTK lemmatization with POS tagging

A more flexible approach is to use a POS tagger to identify each word's grammatical category, then map that category to the format expected by WordNet.

The example downloads the tokenizer, POS tagger, and WordNet resources required by the workflow.

In [25]:
# Example 9: Download resources for tokenization and POS tagging.
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [26]:
# Import the NLTK tools needed for POS-aware lemmatization.
from nltk import pos_tag, word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

# Initialize the lemmatizer.
lemmatizer = WordNetLemmatizer()


def get_wordnet_pos(word):
    """Convert an NLTK POS tag into a WordNet part-of-speech label."""

    tag = pos_tag([word])[0][1]

    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        # Use noun as the fallback category.
        return wordnet.NOUN


# Test POS-aware lemmatization with a sentence.
text = "The children are playing with their toys."
tokens = word_tokenize(text)

for word in tokens:
    if word.isalpha():
        pos = get_wordnet_pos(word)
        lemma = lemmatizer.lemmatize(word, pos=pos)
        print(f"{word:<12} -> {lemma}")

The          -> The
children     -> child
are          -> be
playing      -> play
with         -> with
their        -> their
toys         -> toy


# Part III — Lemmatizing a literary extract from a TXT file

## 9. Working with an external text file

This practical section uses a separate text file named **`alice_extract.txt`**. It contains a different passage from *Alice's Adventures in Wonderland* by Lewis Carroll, taken from Chapter II, **"The Pool of Tears"**.

The passage is provided as a companion file so that the notebook demonstrates a realistic file-reading workflow rather than relying only on text typed directly into a code cell.

**Before running the next cells:** Keep `alice_extract.txt` in the same folder as this notebook.

In [28]:
# Example 1: Read the story extract from the companion TXT file.
from pathlib import Path

# Build the path relative to the notebook's current working directory.
text_file = Path("alice_extract.txt")

with open("alice_extract.txt", encoding="utf-8") as file:
    story_text = file.read()

print(story_text)

ALICE'S ADVENTURES IN WONDERLAND
Lewis Carroll — Chapter II: The Pool of Tears

“Curiouser and curiouser!” cried Alice (she was so much surprised, that for the moment she quite forgot how to speak good English); “now I'm opening out like the largest telescope that ever was! Good-bye, feet!” (for when she looked down at her feet, they seemed to be almost out of sight, they were getting so far off). “Oh, my poor little feet, I wonder who will put on your shoes and stockings for you now, dears? I'm sure I shan't be able! I shall be a great deal too far off to trouble myself about you: you must manage the best way you can;—but I must be kind to them,” thought Alice, “or perhaps they won't walk the way I want to go! Let me see: I'll give them a new pair of boots every Christmas.”

She went on planning to herself how she would manage it. “They must go by the carrier,” she thought; “and how funny it'll seem, sending presents to one's own feet! And how odd the directions will look!”



### 9.1 Lemmatize the story using spaCy

Process the complete extract with spaCy. The output table displays each original token, its part of speech, and its lemma. Punctuation is skipped to keep the results focused on words.

In [29]:
# Example 2: Display each story token and its spaCy lemma.
story_doc = nlp(story_text)

print(f"{'TOKEN':<18} {'PART OF SPEECH':<16} LEMMA")
print("-" * 48)

for token in story_doc:
    if token.is_alpha:
        print(f"{token.text:<18} {token.pos_:<16} {token.lemma_}")

TOKEN              PART OF SPEECH   LEMMA
------------------------------------------------
ALICE              PROPN            ALICE
ADVENTURES         PROPN            ADVENTURES
IN                 ADP              in
WONDERLAND         PROPN            WONDERLAND
Lewis              PROPN            Lewis
Carroll            PROPN            Carroll
Chapter            NOUN             chapter
II                 PROPN            II
The                DET              the
Pool               PROPN            Pool
of                 ADP              of
Tears              PROPN            Tears
Curiouser          PROPN            Curiouser
and                CCONJ            and
curiouser          NOUN             curiouser
cried              VERB             cry
Alice              PROPN            Alice
she                PRON             she
was                AUX              be
so                 ADV              so
much               ADV              much
surprised          ADJ        

### 9.2 Create a cleaned, lemmatized version

Remove stop words and punctuation, then join the remaining lemmas into a single string. This is a common preprocessing step before tasks such as text analysis or vectorization.

In [30]:
# Example 3: Clean and lemmatize the complete story extract.
story_lemmas = [
    token.lemma_.lower()
    for token in story_doc
    if token.is_alpha and not token.is_stop
]

lemmatized_story = " ".join(story_lemmas)

print("LEMMATIZED STORY EXTRACT")
print("-" * 28)
print(lemmatized_story)

LEMMATIZED STORY EXTRACT
----------------------------
alice adventures wonderland lewis carroll chapter ii pool tears curiouser curiouser cry alice surprised moment forget speak good english open like large telescope good bye foot look foot sight get far oh poor little foot wonder shoe stocking dear sure shall able shall great deal far trouble manage good way kind think alice will walk way want let new pair boot christmas go plan manage carrier think funny send present foot odd direction look


### 9.3 Save the lemmatized output to a new TXT file

The processed text can be saved for later use. This creates `alice_extract_lemmatized.txt` in the notebook's current working directory.

In [31]:
# Example 4: Save the lemmatized story to a separate text file.
output_file = Path("alice_extract_lemmatized.txt")

output_file.write_text(lemmatized_story, encoding="utf-8")

print(f"Lemmatized text saved to: {output_file.resolve()}")

Lemmatized text saved to: /content/alice_extract_lemmatized.txt


## 10. Key observations

- Lemmatization converts words into meaningful dictionary base forms.
- spaCy uses linguistic annotations from its language model to determine lemmas.
- NLTK's `WordNetLemmatizer` can use noun, verb, adjective, and adverb categories.
- NLTK's default part of speech is noun, so verbs often require `pos="v"`.
- POS tagging can improve NLTK lemmatization by selecting a more suitable grammatical category.
- Stop-word and punctuation removal are optional preprocessing choices; they should depend on the goal of the task.
- Reading a `.txt` file allows the same preprocessing workflow to be applied to longer external documents.

## 11. Conclusion

This notebook explored lemmatization using spaCy and NLTK, from individual words and sentences to a complete literary extract stored in a text file. It demonstrated why part-of-speech information matters, how to remove common text elements, and how to save the processed result. Lemmatization is an important preprocessing technique when meaningful base forms are preferred over rule-based stems.

---